In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('buildPyspark').getOrCreate()

In [0]:
#Create a DataFrame: Create a DataFrame from a list of dictionaries with an explicit schema.

dataDict = [
    {"Name":"Sagar","Age":24,"Salary":50000},
    {"Name":"Shyam","Age":24,"Salary":60000},
    {"Name":"Raghav","Age":24,"Salary":30000},
    {"Name":"Deepak","Age":24,"Salary":56000},
    {"Name":"Pranjal","Age":24,"Salary":58000},
    {"Name":"Kartik","Age":24,"Salary":20000},
    
]

df = spark.createDataFrame(dataDict)
df.show()
display(df)


+---+-------+------+
|Age|   Name|Salary|
+---+-------+------+
| 24|  Sagar| 50000|
| 24|  Shyam| 60000|
| 24| Raghav| 30000|
| 24| Deepak| 56000|
| 24|Pranjal| 58000|
| 24| Kartik| 20000|
+---+-------+------+



Age,Name,Salary
24,Sagar,50000
24,Shyam,60000
24,Raghav,30000
24,Deepak,56000
24,Pranjal,58000
24,Kartik,20000


In [0]:
#Select and rename : Select required columns and rename them to a standard naming convention.

from pyspark.sql.functions import col

new_df = df.select(col("Name").alias("Emp_name"),
                   col("Age").alias("emp_age"),
                   col("Salary").alias("Emp_Salary")
                   )

display(new_df)

Emp_name,emp_age,Emp_Salary
Sagar,24,50000
Shyam,24,60000
Raghav,24,30000
Deepak,24,56000
Pranjal,24,58000
Kartik,24,20000


In [0]:
#Filter records : Filter records having salary greter than 50000

filter_df = df.filter(col("Salary") > 50000)

display(filter_df)


Age,Name,Salary
24,Shyam,60000
24,Deepak,56000
24,Pranjal,58000


In [0]:
#withColumn : Create a Yearly_Salary of each emp

withYearlySal_df = df.withColumn("Yearly_Salary",col("Salary") * 12 )

display(withYearlySal_df)

Age,Name,Salary,Yearly_Salary
24,Sagar,50000,600000
24,Shyam,60000,720000
24,Raghav,30000,360000
24,Deepak,56000,672000
24,Pranjal,58000,696000
24,Kartik,20000,240000


In [0]:
#Null handling : Identify null counts by column and fill or remove missing values appropriately.

data = [
    {"Name": "Sagar",   "Age": 24,   "Salary": 50000, "City": "Ahmedabad"},
    {"Name": "Shyam",   "Age": 25,   "Salary": 60000, "City": "Surat"},
    {"Name": "Raghav",  "Age": None, "Salary": 30000, "City": "Vadodara"},
    {"Name": "Deepak",  "Age": 27,   "Salary": None,  "City": "Rajkot"},
    {"Name": "Pranjal", "Age": 26,   "Salary": 58000, "City": None},
    {"Name": "Kartik",  "Age": 23,   "Salary": 20000, "City": "Surat"},
    {"Name": "Amit",    "Age": None, "Salary": 45000, "City": "Mumbai"},
    {"Name": "Rahul",   "Age": 29,   "Salary": 70000, "City": None},
    {"Name": "Neha",    "Age": 28,   "Salary": None,  "City": "Pune"},
    {"Name": "Priya",   "Age": 24,   "Salary": 65000, "City": "Delhi"}
]

df = spark.createDataFrame(data)

# display(df)

#filling value at none 
new_df = df.fillna({"Name" : "NA", "Age" : 0, "Salary" : 0, "City" : "Unknown"})
# display(new_df)

#Null Count

from pyspark.sql.functions import sum 

display(df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]))

Age,City,Name,Salary
2,2,0,2


In [0]:
#Null handling and total null count code

display(df.select([
    sum(col("Age").isNull().cast("int")).alias(c)
    for c in df.columns
]
))

handle_df = df.fillna({"Age" : 0,"City":"Unknown"})
display(handle_df)

display(df.dropna())

Age,City,Name,Salary
2,2,2,2


Age,City,Name,Salary
24,Ahmedabad,Sagar,50000
25,Surat,Shyam,60000
0,Vadodara,Raghav,30000
27,Rajkot,Deepak,null
26,Unknown,Pranjal,58000
23,Surat,Kartik,20000
0,Mumbai,Amit,45000
29,Unknown,Rahul,70000
28,Pune,Neha,null
24,Delhi,Priya,65000


Age,City,Name,Salary
24,Ahmedabad,Sagar,50000
25,Surat,Shyam,60000
23,Surat,Kartik,20000
24,Delhi,Priya,65000


In [0]:
#Remove duplicates : Remove exact duplicate rows and explain how dropDuplicates differs when given a subset of columns.

data = [
    {"ID": 1, "Name": "Sagar", "City": "Ahmedabad", "Salary": 50000},
    {"ID": 2, "Name": "Rahul", "City": "Surat", "Salary": 60000},
    {"ID": 3, "Name": "Sagar", "City": "Ahmedabad", "Salary": 50000},
    {"ID": 4, "Name": "Priya", "City": "Pune", "Salary": 55000},
    {"ID": 5, "Name": "Rahul", "City": "Surat", "Salary": 60000},
    {"ID": 6, "Name": "Amit", "City": "Mumbai", "Salary": 45000},
    {"ID": 7, "Name": "Sagar", "City": "Delhi", "Salary": 70000},
    {"ID": 8, "Name": "Neha", "City": "Pune", "Salary": 50000},
    {"ID": 9, "Name": "Priya", "City": "Pune", "Salary": 55000},
    {"ID": 10, "Name": "Amit", "City": "Mumbai", "Salary": 45000}
]

df = spark.createDataFrame(data)

# display(df)

display(df.dropDuplicates(["Name","city"]))

# -------------------------------------
# dropDuplicates() checks all columns. Rows are duplicates only when every column matches.
# dropDuplicates(["Name"]) checks only the Name column. If multiple rows share a name, only one is retained.
# When duplicate rows have different values in other columns, Spark does not guarantee which row is retained.
# -------------------------------------

City,ID,Name,Salary
Ahmedabad,1,Sagar,50000
Surat,2,Rahul,60000
Pune,4,Priya,55000
Mumbai,6,Amit,45000
Delhi,7,Sagar,70000
Pune,8,Neha,50000


In [0]:
#Conditional columns : Create a transaction status category using when and otherwise.

data = [
    {"TransactionID": 101, "Amount": 500,   "Status": "Completed"},
    {"TransactionID": 102, "Amount": 15000, "Status": "Pending"},
    {"TransactionID": 103, "Amount": 2500,  "Status": "Failed"},
    {"TransactionID": 104, "Amount": 8000,  "Status": "Completed"},
    {"TransactionID": 105, "Amount": 50000, "Status": "Pending"},
    {"TransactionID": 106, "Amount": 1200,  "Status": "Completed"},
    {"TransactionID": 107, "Amount": 30000, "Status": "Failed"},
    {"TransactionID": 108, "Amount": 700,   "Status": "Pending"},
    {"TransactionID": 109, "Amount": 12000, "Status": "Completed"},
    {"TransactionID": 110, "Amount": 4000,  "Status": "Failed"}
]

df = spark.createDataFrame(data)

from pyspark.sql.functions import col, when

new_df = df.withColumn(
    "StatusCategory",
    when(col("status") == "Completed", "Succesfull Done")
    .when(col("status") == "Pending" , "Status Pending State")
    .when(col("status") == "Failed" , "Status Failed")
    .otherwise("Unknown")
)

display(new_df)

Amount,Status,TransactionID,StatusCategory
500,Completed,101,Succesfull Done
15000,Pending,102,Status Pending State
2500,Failed,103,Status Failed
8000,Completed,104,Succesfull Done
50000,Pending,105,Status Pending State
1200,Completed,106,Succesfull Done
30000,Failed,107,Status Failed
700,Pending,108,Status Pending State
12000,Completed,109,Succesfull Done
4000,Failed,110,Status Failed


In [0]:
data = [
    {"OrderID": 101, "Customer": "Sagar",  "OrderDate": "2026-01-05", "Revenue": 2500},
    {"OrderID": 102, "Customer": "Rahul",  "OrderDate": "2026-01-10", "Revenue": 4000},
    {"OrderID": 103, "Customer": "Sagar",  "OrderDate": "2026-01-20", "Revenue": 1500},
    {"OrderID": 104, "Customer": "Priya",  "OrderDate": "2026-01-25", "Revenue": 3000},
    {"OrderID": 105, "Customer": "Rahul",  "OrderDate": "2026-02-03", "Revenue": 2000},
    {"OrderID": 106, "Customer": "Sagar",  "OrderDate": "2026-02-12", "Revenue": 3500},
    {"OrderID": 107, "Customer": "Priya",  "OrderDate": "2026-02-15", "Revenue": 4500},
    {"OrderID": 108, "Customer": "Rahul",  "OrderDate": "2026-02-20", "Revenue": 1000},
    {"OrderID": 109, "Customer": "Sagar",  "OrderDate": "2026-03-01", "Revenue": 5000},
    {"OrderID": 110, "Customer": "Priya",  "OrderDate": "2026-03-10", "Revenue": 2500}
]

df = spark.createDataFrame(data)

from pyspark.sql.functions import to_date
#This converts the existing column from string to DateType. It does not create another column.
df = df.withColumn("OrderDate", to_date("OrderDate"))
# display(df)

#Aggregations : Calculate total revenue and order count by customer and month.
from pyspark.sql.functions import sum, count, date_format, col
display(df.select(
        sum("Revenue").alias("Total_revenue")
))

result_df = df.withColumn(
    "Month", date_format(col("OrderDate"), "yyyy-MM")
).groupBy(
    "Customer", "Month"
).agg(
    sum("Revenue").alias("TotalRevenue"),
    count("OrderID").alias("OrderCount")
).orderBy(
    col("Customer").asc(),
    col("Month").asc()
)

display(result_df)

Total_revenue
29500


Customer,Month,TotalRevenue,OrderCount
Priya,2026-01,3000,1
Priya,2026-02,4500,1
Priya,2026-03,2500,1
Rahul,2026-01,4000,1
Rahul,2026-02,3000,2
Sagar,2026-01,4000,2
Sagar,2026-02,3500,1
Sagar,2026-03,5000,1


Typical PySpark operation order
1. withColumn() — create or modify columns.
2. filter() — filter rows.
3. groupBy() — group rows.
4. agg() — calculate aggregations.
5. select() — choose the final columns, if needed.
6. orderBy() — sort the result.
7. show() / display() — view the output.